# 01 - Construcción del Dataset

Este notebook transforma el dataset original de MMHS150K en una versión limpia, estructurada y lista para su uso en tareas de clasificación de lenguaje.

En este paso se realizan las operaciones necesarias para convertir los datos crudos en un archivo procesado que pueda utilizarse en el resto del proyecto. Se leen los registros del JSON principal, se incorporan los splits oficiales de train/val/test, se resuelven las etiquetas en dos esquemas diferentes (multiclase y binario), se limpia de forma mínima el texto de cada tweet y se filtran las muestras que no cumplen con los requisitos básicos para modelar.

Salida final:
- Un único archivo CSV en `data/processed/dataset.csv` con las columnas `id`, `text`, `label_multiclass`, `label_binary` y `split`.

Importante:
- Este notebook no realiza entrenamiento de modelos ni análisis exploratorio de datos (EDA).
- A partir de aquí, todo el flujo posterior del proyecto debe trabajar únicamente sobre el archivo generado.

## 1. Configuración e importación de librerías

In [89]:
import json
import re
import os
from collections import Counter

import pandas as pd

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"


## 2. Leer el JSON principal (`MMHS150K_GT.json`)

In [90]:
with open(f"{RAW_DIR}/MMHS150K_GT.json", encoding="utf-8") as f:
    gt = json.load(f)

print(f"Total de tweets en el JSON: {len(gt)}")

# Inspeccionar una muestra para validar el formato esperado
sample_id = next(iter(gt))
print("\nEjemplo de entrada:")
print(json.dumps(gt[sample_id], indent=2, ensure_ascii=False))


Total de tweets en el JSON: 149823

Ejemplo de entrada:
{
  "img_url": "http://pbs.twimg.com/tweet_video_thumb/D3gi9MHWAAAgfl7.jpg",
  "labels": [
    4,
    1,
    3
  ],
  "tweet_url": "https://twitter.com/user/status/1114679353714016256",
  "tweet_text": "@FriskDontMiss Nigga https://t.co/cAsaLWEpue",
  "labels_str": [
    "Religion",
    "Racist",
    "Homophobe"
  ]
}


## 3. Validación inicial

Antes de construir el dataset final, verificamos la calidad e integridad de los datos crudos. Esta etapa sirve para detectar problemas tempranos en los registros, como textos faltantes, entradas vacías o inconsistencias en las anotaciones. Si se encuentra algún problema aquí, es más sencillo corregirlo antes de generar el dataset que se usará en los pasos posteriores.


In [91]:
n_total = len(gt)
n_missing_text = sum(1 for v in gt.values() if not v.get("tweet_text"))
n_empty_text = sum(1 for v in gt.values() if v.get("tweet_text", "").strip() == "")

print(f"Total de registros: {n_total}")
print(f"Registros sin campo 'tweet_text': {n_missing_text}")
print(f"Registros con 'tweet_text' vacío tras strip(): {n_empty_text}")

# Distribución cruda de las anotaciones (antes de resolver votación)
all_raw_labels = [label for v in gt.values() for label in v["labels"]]
print("\nDistribución cruda de anotaciones individuales (0-5):")
print(Counter(all_raw_labels))


Total de registros: 149823
Registros sin campo 'tweet_text': 0
Registros con 'tweet_text' vacío tras strip(): 0

Distribución cruda de anotaciones individuales (0-5):
Counter({0: 312039, 1: 63543, 5: 31548, 2: 22805, 3: 16932, 4: 2607})


## 4. Leer los splits oficiales (train / val / test)

In [92]:
def read_ids(path):
    with open(path, encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())

train_ids = read_ids(f"{RAW_DIR}/splits/train_ids.txt")
val_ids   = read_ids(f"{RAW_DIR}/splits/val_ids.txt")
test_ids  = read_ids(f"{RAW_DIR}/splits/test_ids.txt")

print(f"train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")

def get_split(tweet_id):
    if tweet_id in train_ids:
        return "train"
    if tweet_id in val_ids:
        return "val"
    if tweet_id in test_ids:
        return "test"
    return None


train: 134823 | val: 5000 | test: 10000


### ⚠️ Hallazgo de calidad de datos: splits duplicados

Durante la validación inicial de este notebook se detectó que, en la primera
descarga del dataset, `train_ids.txt` y `test_ids.txt` contenían exactamente
los mismos 10,000 IDs (probablemente por un error de copiado de archivos
durante la descarga/descompresión). Esto causaba que ningún tweet quedara
asignado al split `test`, ya que la función `get_split()` revisa `train_ids`
primero y el solapamiento absorbía todos los IDs de test bajo `train`.

**Diagnóstico:** se verificó mediante intersección de conjuntos
(`train_ids & test_ids`), confirmando un overlap de 10,000/10,000.

**Solución:** se volvió a descargar el dataset, obteniendo splits correctos
y disjuntos:

```python
print(f"train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")
# train: 134823 | val: 5000 | test: 10000

print(f"Overlap train/test: {len(train_ids & test_ids)}")  # 0
print(f"Overlap train/val:  {len(train_ids & val_ids)}")   # 0
print(f"Overlap val/test:   {len(val_ids & test_ids)}")    # 0
```

Se documenta este proceso como evidencia de la validación de integridad
realizada sobre los datos crudos antes de construir el dataset final.

In [93]:
print(f"train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")

json_keys = set(gt.keys())
print(f"train en JSON: {len(train_ids & json_keys)} / {len(train_ids)}")
print(f"val en JSON:   {len(val_ids & json_keys)} / {len(val_ids)}")
print(f"test en JSON:  {len(test_ids & json_keys)} / {len(test_ids)}")

print(f"Overlap train/test: {len(train_ids & test_ids)}")
print(f"Overlap train/val:  {len(train_ids & val_ids)}")
print(f"Overlap val/test:   {len(val_ids & test_ids)}")

print("\nDistribucion final por split:")
print(df["split"].value_counts())

train: 134823 | val: 5000 | test: 10000
train en JSON: 134823 / 134823
val en JSON:   5000 / 5000
test en JSON:  10000 / 10000
Overlap train/test: 0
Overlap train/val:  0
Overlap val/test:   0

Distribucion final por split:
split
train    134823
test      10000
val        5000
Name: count, dtype: int64


## 5. Resolver etiquetas: esquema multiclase y esquema binario

En esta etapa se convierten las anotaciones originales en dos etiquetas de referencia que serán útiles para distintos tipos de experimentos. El esquema multiclase conserva las seis categorías originales y permite evaluar el modelo en una tarea más específica, mientras que el esquema binario simplifica el problema a una decisión de “hate” vs “not hate”, lo que suele ser más directo para clasificación básica.

- **Multiclase:** se aplica una votación mayoritaria directa sobre las 6 categorías. Si no existe una categoría con mayoría clara, la muestra se descarta porque no hay consenso suficiente para asignar una etiqueta fiable.
- **Binario:** cada voto se convierte en una etiqueta binaria (`0` → NotHate, `1-5` → Hate) y luego se aplica la misma lógica de votación. Como hay solo dos clases y tres votos, este esquema suele tener una decisión más estable.


In [94]:
LABEL_NAMES = {
    0: "NotHate",
    1: "Racist",
    2: "Sexist",
    3: "Homophobe",
    4: "Religion",
    5: "OtherHate",
}

def majority_vote_multiclass(labels):
    counts = Counter(labels)
    top_label, top_count = counts.most_common(1)[0]
    if top_count < 2:
        return None  # empate perfecto, sin mayoria -> se descarta
    return top_label

def majority_vote_binary(labels):
    binary_votes = [0 if l == 0 else 1 for l in labels]
    counts = Counter(binary_votes)
    return counts.most_common(1)[0][0]  # siempre existe mayoria


## 6. Limpieza mínima del texto

In [95]:
def clean_text(text):
    text = re.sub(r"RT\s+@?\w+:?", "", text)   # prefijo RT + usuario retuiteado
    text = re.sub(r"http\S+", "", text)          # URLs
    text = re.sub(r"\s+", " ", text).strip()      # espacios repetidos
    return text


## 7. Construir el DataFrame final

Aquí se ensamblan todas las decisiones tomadas en los pasos anteriores para generar una tabla uniforme lista para usar. Cada fila representa un tweet valido, con su identificador, texto limpio, etiqueta en el esquema multiclase, etiqueta en el esquema binario y el split al que pertenece. Esta tabla es la salida principal del notebook y será la base del resto del proyecto.


In [96]:
rows = []
for tweet_id, entry in gt.items():
    text = entry.get("tweet_text", "")
    if not text or not text.strip():
        continue  # descartamos tweets sin texto, no aportan a un modelo de NLP

    split = get_split(tweet_id)
    if split is None:
        continue  # tweet no pertenece a ningun split oficial

    labels = entry["labels"]

    rows.append({
        "id": tweet_id,
        "text": clean_text(text),
        "label_multiclass": majority_vote_multiclass(labels),
        "label_multiclass_name": LABEL_NAMES.get(majority_vote_multiclass(labels)),
        "label_binary": majority_vote_binary(labels),
        "split": split,
    })

df = pd.DataFrame(rows)
df.shape


(149823, 6)

## 8. Verificación del dataset construido

Antes de guardar el resultado, es importante revisar que el DataFrame tenga el formato esperado y que las distribuciones de las etiquetas sean consistentes. Esta verificación ayuda a confirmar que el proceso de construcción del dataset se ejecutó correctamente y que no se introdujeron errores visibles durante la transformación.


In [97]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 149823 entries, 0 to 149822
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     149823 non-null  str    
 1   text                   149823 non-null  str    
 2   label_multiclass       138109 non-null  float64
 3   label_multiclass_name  138109 non-null  str    
 4   label_binary           149823 non-null  int64  
 5   split                  149823 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 6.9 MB


In [98]:
df.head()


,id,text,label_multiclass,label_multiclass_name,label_binary,split
0,1114679353714016256,@FriskDontMiss Nigga,NaN,NaN,1,train
1,1063020048816660480,My horses are retarded,5.0,OtherHate,1,train
2,1108927368075374593,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL S...,0.0,NotHate,0,train
3,1114558534635618305,I ran into this HOLY NIGGA TODAY 😭😭😭😭,0.0,NotHate,0,train
4,1035252480215592966,“EVERYbody calling you Nigger now!”,1.0,Racist,1,val


In [99]:
n_descartadas_multiclass = df["label_multiclass"].isna().sum()
print(f"Muestras descartadas por empate (esquema multiclase): {n_descartadas_multiclass} "
      f"({n_descartadas_multiclass / len(df) * 100:.2f}%)")

print("\nDistribucion multiclase:")
print(df["label_multiclass_name"].value_counts(dropna=False))

print("\nDistribucion binaria:")
print(df["label_binary"].map({0: "NotHate", 1: "Hate"}).value_counts())

print("\nDistribucion por split:")
print(df["split"].value_counts())


Muestras descartadas por empate (esquema multiclase): 11714 (7.82%)

Distribucion multiclase:
label_multiclass_name
NotHate      112844
Racist        11926
NaN           11714
OtherHate      5811
Homophobe      3870
Sexist         3495
Religion        163
Name: count, dtype: int64

Distribucion binaria:
label_binary
NotHate    112852
Hate        36971
Name: count, dtype: int64

Distribucion por split:
split
train    134823
test      10000
val        5000
Name: count, dtype: int64


## 9. Guardar dataset procesado

A partir de este punto, todo el resto del proyecto (EDA, entrenamiento, evaluación) trabajará únicamente sobre `dataset.csv`, sin volver a leer el JSON original.

In [100]:
df.to_csv(f"{PROCESSED_DIR}/dataset.csv", index=False)
print(f"Guardado en {PROCESSED_DIR}/dataset.csv - {len(df)} filas")


Guardado en ../data/processed/dataset.csv - 149823 filas
